# Day 2 · 06. Adaptive RAG — 질문 복잡도에 따른 검색 경로 분기

## 학습 목표
- 쿼리의 성격에 따라 검색 경로를 **No-retrieval / Single-step / Multi-step** 세 갈래로 분기하는 adaptive RAG를 설계할 수 있다.
- Pydantic schema + LLM 구조화 출력으로 **쿼리 라우터**를 구현할 수 있다.
- LangGraph `StateGraph`의 `add_conditional_edges`로 라우팅 로직을 명시적 그래프로 표현할 수 있다.
- Self-RAG(검색 **후** 평가)와 Adaptive RAG(검색 **전** 경로 결정)의 **운영·감사 관점 차이**를 비교할 수 있다.

## 핵심 키워드
`adaptive RAG` · `query router` · `query complexity classifier` · `conditional edge (LangGraph)` · `multi-hop retrieval`

## 이 노트북의 위치
```
05 (Self-RAG, 검색 후 재시도)   →   06 (질문 분류로 경로를 입구에서 결정)
```
05에서는 검색을 먼저 한 뒤 품질을 사후 평가해 필요하면 재작성·재검색했다. 여기서는 시점을 한 단계 앞으로 당긴다 — **입구에서 질문 유형부터 분류하고, 그에 맞는 경로로만 자원을 쓴다**. 단일 경로 안에서는 Self-RAG 스타일의 `grade → rewrite` 루프를 그대로 재활용한다.

## 1. Adaptive RAG 개념 — "질문마다 다른 도구를 쓴다"

### 🚑 직관: 응급실 triage nurse

환자가 병원 접수창구에 도착하면, 의사 앞까지 가기 전에 **triage nurse(분류 간호사)** 가 3초 안에 아래 중 한 줄로 보낸다.

| 환자 상태 | 배정되는 자원 | RAG 대응 경로 |
|-----------|----------------|----------------|
| 가벼운 감기·단순 문의 | 귀가 지도, 비처방 안내 (검사 없음) | `no_retrieval` — LLM 일반 지식만 |
| 명확한 단일 증상 | 해당 과 외래 1회 진료 | `single_retrieval` — 약관 1회 조회 |
| 복합 증상·다학제 | 여러 검사·과 협진 | `multi_retrieval` — 다단계 검색·합성 |

Adaptive RAG는 **똑같은 triage 원칙을 RAG 파이프라인 입구에 둔다**. "모든 질문을 똑같이 검색 돌리지 말고, 질문 성격에 맞는 경로로만 자원을 쓰자"는 아이디어다.

### 논문 배경 — Jeong et al., 2024

[*Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity*](https://arxiv.org/abs/2403.14403) 가 바로 이 3갈래 분류를 정식으로 제안한 논문이다. 실험에서 **같은 정확도 대비 비용 최대 40% 절감**을 보고했다.

| 복잡도 | 예시 | 파이프라인 |
|--------|------|----------|
| **A. No-retrieval** | "안녕하세요", "당신은 누구" | LLM만 사용 (검색 없음) |
| **B. Single-step** | "청약철회 기간은?" | retrieval 1회 + 생성 |
| **C. Multi-step** | "A사 자회사가 발행한 상품의 약관상 철회 기간은?" | 다단계 검색 + 중간 합성 |

### 왜 "초기 1회 분기"인가?

> 🧠 **비교 — 05 Self-RAG와의 관계**: Self-RAG는 **일단 검색하고 나서** 관련성을 판정, 부족하면 쿼리를 다시 써 재검색했다. Adaptive RAG는 **검색 전에 질문을 먼저 분류**해 경로 자체를 갈라놓는다. 두 접근은 배타적이지 않고 — 실무에서는 **Adaptive 로 경로를 나눈 뒤, single 경로 안에서 Self-RAG 스타일의 grade·rewrite 루프를 돌리는** 조합이 흔하다. 이 노트북도 그 조합으로 구현한다.

- **검색 비용 절약**: 쉬운 질문에 retrieval 돌릴 필요가 없다.
- **복잡 질문 정확도 향상**: 한 번의 검색으로 부족한 정보를 여러 번 나눠 조회한다.
- **감사 친화적**: 경로가 유한·예측 가능 → "왜 이 답이 나왔는지" 재구성이 쉽다.

```mermaid
flowchart TD
    Q[질문] --> R{classify_query}
    R -- no_retrieval --> D[direct_answer]
    R -- single_retrieval --> S[single_rag]
    R -- multi_retrieval --> M[multi_hop_rag]
    D --> G[generate]
    S --> G
    M --> G
    G --> END([최종 답변])
```

In [ ]:
# 공통 세팅
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

## 2. 약관 벡터스토어 준비 (본 노트북 내 자기완결)

벤치마크 질문으로 라우터가 올바른 경로를 선택하는지 평가한다. 코퍼스는 단독 재현이 가능하도록 노트북 내부에 포함한다.

### 왜 9개 문서인가?

| 목적 | 문서 | 촉발되는 경로 |
|------|------|----------------|
| **단일 사실 질의** 소재 | `제1조·제2조·제5조·제8조·제10조·제14조·제17조` (7개) | `single_retrieval` |
| **엔티티 체인** 소재 | `그룹구조(삼성 계열사)` + `상품안내(삼성카드 청약철회)` (2개) | `multi_retrieval` |
| **검색 불필요** 예시 | (문서 없이) "안녕", "오늘 날씨" 같은 일상 질의 | `no_retrieval` |

→ **3-way 경로를 모두 촉발할 수 있는 최소 집합**. 실무에서는 수만 청크로 확장되지만, 학습용으로는 9개로 라우팅 로직을 감상할 수 있다.

### `metadata.article` 태그는 왜 붙는가?

`제1조`, `그룹구조` 같은 **사람이 읽을 수 있는 라벨**을 metadata에 넣어두면, 나중에 `trace` 로그에 "어떤 조항이 검색되었는지" 한눈에 표시된다. 운영 환경에서도 원문 URL·페이지 번호·모델 버전 등을 같은 자리에 넣어 **감사 추적성**을 확보한다.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

E_FINANCE_TERMS = [
    Document(page_content='제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함으로써 이용자의 권익 보호와 거래의 안전을 확보함을 목적으로 한다.', metadata={'article': '제1조'}),
    Document(page_content='제2조(정의) 전자금융거래란 금융회사 또는 전자금융업자가 전자적 장치를 통해 제공하는 금융상품 및 서비스를 자동화된 방식으로 이용하는 거래를 말한다.', metadata={'article': '제2조'}),
    Document(page_content='제5조(청약철회) ① 이용자는 계약 체결 후 14일 이내에 서면 또는 전자문서로 청약을 철회할 수 있다.', metadata={'article': '제5조'}),
    Document(page_content='제8조(수수료) 전자금융거래 이용 수수료는 거래 유형별로 사전에 안내되며, 수수료 인상 시 적용일 30일 전에 공지해야 한다.', metadata={'article': '제8조'}),
    Document(page_content='제10조(분실신고) ① 이용자는 접근매체의 분실·도난 시 즉시 고객센터 또는 가장 가까운 영업점에 신고하여야 한다.', metadata={'article': '제10조'}),
    Document(page_content='제14조(이의신청) ① 이용자는 거래 내역에 이의가 있을 때 거래일로부터 1년 이내에 이의를 신청할 수 있다. ② 금융회사는 이의신청 접수일로부터 15영업일 이내에 결과를 통지한다.', metadata={'article': '제14조'}),
    Document(page_content='제17조(서비스 이용시간) 전자금융거래 서비스는 24시간 이용을 원칙으로 하되, 시스템 점검·장애·운영상 필요에 따라 일시적으로 제한될 수 있다.', metadata={'article': '제17조'}),
    # Multi-hop 시나리오를 위한 보조 문서
    Document(page_content='삼성전자의 자회사로는 삼성카드와 삼성생명이 있으며, 이들은 각각 신용카드 및 보험 상품을 발행한다.', metadata={'article': '그룹구조'}),
    Document(page_content='삼성카드가 발행하는 신용카드 상품은 전자금융거래 표준약관을 준용하며, 청약철회는 제5조에 따라 14일 이내에 가능하다.', metadata={'article': '상품안내'}),
]

vs = Chroma.from_documents(
    E_FINANCE_TERMS,
    embedding=get_embeddings(),
    collection_name='efinance_adaptive',
)
retriever = vs.as_retriever(search_kwargs={'k': 3})
llm = get_chat_model(temperature=0)
print(f'약관 {len(E_FINANCE_TERMS)}개 인덱싱 완료')

## 3. 쿼리 복잡도 분류기 (라우터)

LLM에게 `route` 필드를 가진 Pydantic 객체를 생성하게 해 **구조화 출력**을 얻는다. 파싱 실패가 없고 다운스트림 `add_conditional_edges`에 그대로 사용할 수 있다.

### 경계선 휴리스틱 — 실무 판정 기준

| 경로 | 체크리스트 | 예시 |
|------|-----------|------|
| `no_retrieval` | ① 약관·도메인 지식 없이 답이 되는가? ② 인사·시간·날씨·자기소개인가? | "안녕", "오늘 날씨 어때", "너는 누구야" |
| `single_retrieval` | ① **하나의 조항/청크**로 답이 완결되는가? ② 질문의 핵심이 명사 1~2개인가? | "청약철회 기간은?", "수수료 공지 주기?" |
| `multi_retrieval` | ① 엔티티 A→B→C 체인 추적이 필요한가? ② 소유격/한정절("의", "~가 발행한")이 있는가? ③ 명사 2개 이상을 이어야 하는가? | "삼성전자 자회사가 발행한 상품의 청약철회 기간?" |

> 💡 **경계선이 애매할 때**: "이 질문 하나의 조항만으로 답할 수 있나?"를 스스로 먼저 시도해 보라. 한 조항으로 답이 끊기면 single, 두 조항을 이어붙여야 하면 multi.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class RouteDecision(BaseModel):
    """질문을 세 가지 경로 중 하나로 분류한다."""
    route: Literal['no_retrieval', 'single_retrieval', 'multi_retrieval'] = Field(
        description=(
            'no_retrieval: 일상 대화/인사/LLM 일반 지식으로 충분. '
            'single_retrieval: 약관 1회 조회로 답할 수 있는 단일 사실 질의. '
            'multi_retrieval: 2개 이상의 문서를 연결해야 답할 수 있는 다단계 질의.'
        )
    )
    reasoning: str = Field(description='1문장으로 분류 근거를 밝힌다.')

router_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 금융 도메인 RAG 시스템의 쿼리 라우터입니다.\n'
     '질문을 보고 세 경로 중 하나로 분류하세요.\n'
     '- 일상 대화(인사, 사소한 잡담)는 no_retrieval.\n'
     '- 약관 한 조항만 보면 답할 수 있으면 single_retrieval.\n'
     '- 엔티티 연결(예: 자회사→상품→약관)이 필요하면 multi_retrieval.'),
    ('user', '{question}'),
])

router = router_prompt | llm.with_structured_output(RouteDecision)

# 1) 기본 동작 확인 — 각 경로가 확실한 예시
print('=== 확실한 경계 ===')
for q in ['안녕', '청약철회 기간은?', '삼성전자 자회사가 발행한 상품의 철회 기간은?']:
    d = router.invoke({'question': q})
    print(f'  {q!r:55s} → {d.route}  ({d.reasoning})')

# 2) 경계가 애매한 예시 — 휴리스틱의 한계를 체감
print('\n=== 경계가 애매한 예 — 라우터의 선택을 관찰 ===')
ambiguous = [
    '전자금융거래가 뭔지 한 줄로만 설명해줘',           # no vs single?
    '분실 신고하면 수수료도 바로 환불되나?',              # single vs multi?
    '삼성카드 청약철회 기간은 며칠인가?',                  # single vs multi?
]
for q in ambiguous:
    d = router.invoke({'question': q})
    print(f'  {q!r:55s} → {d.route}')
    print(f'    └ 근거: {d.reasoning}')

## 4. LangGraph StateGraph — 조건부 분기

### 🎒 3요소 비유 — "배낭 여행"

| 요소 | 비유 | 코드 |
|------|------|------|
| **State** | 질문이 노드를 거치며 들고 다니는 **배낭** (= 공용 메모) | `TypedDict` |
| **Node** | 배낭을 열어 한 가지 일을 하고 다시 닫는 **함수** | `def node(state) -> state` |
| **Edge** | 다음 노드로 가는 **도로**. `add_conditional_edges`는 갈림길의 **표지판** | `add_edge`, `add_conditional_edges` |

### State 필드 — 어느 노드가 어느 필드를 쓰나

| State 필드 | 초기값 | `classify_query` | `single_rag` | `multi_hop_rag` | `grade_documents` | `transform_query` | `generate` |
|-----------|--------|------------------|--------------|-----------------|-------------------|-------------------|------------|
| `question` | 입력 | 읽음 | 읽음 | 읽음 | — | **재쓰기** | 읽음 |
| `route` | `""` | **쓰기** | — | — | — | — | — |
| `documents` | `[]` | — | **쓰기** | **쓰기** | 읽음 | — | 읽음 |
| `generation` | `""` | — | — | — | — | — | **쓰기** |
| `trace` | `[]` | append | append | append | append | append | append |
| `needs_rewrite` | `False` | — | — | — | **쓰기** | 읽음 | — |
| `rewrite_count` | `0` | — | — | — | 읽음 | **+1** | — |

> 🏷️ **확장 원칙**: State에 필드를 추가하기만 하면 모든 노드가 접근 가능하다. 처음엔 필수 필드만 두고, 품질 보강(여기서는 `grade_documents` → `transform_query`) 같은 추가 로직이 필요해질 때 필드를 **늘리는 방향으로**만 리팩터링한다.

### 노드·엣지 구조

```
   ┌───────────┐
   │ classify  │
   └─────┬─────┘
         │  조건부 분기
   ┌─────┼──────┐
   ▼     ▼      ▼
 direct single  multi
         │      │
         ▼      │
   grade_documents
         │
   ┌─────┴─────┐
   ▼(무관)    ▼(관련 있음)
 transform    generate ◀── direct, multi도 여기로 수렴
 _query
   │
   └─→ single 로 복귀 (최대 1회)
```

> 🛡️ **무한 루프 방지**: `rewrite_count` 가 1을 초과하면 transform_query 경로를 끊고 바로 generate로 탈출한다.

In [ ]:
from typing import TypedDict
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

class AdaptiveState(TypedDict):
    question: str
    route: str
    documents: list[Document]
    generation: str
    trace: list[str]          # 경유한 노드 이름을 누적
    needs_rewrite: bool       # grade_documents 가 판정 → transform_query 분기 조건
    rewrite_count: int        # 재작성 횟수 상한(1)


def classify_query(state: AdaptiveState) -> AdaptiveState:
    d = router.invoke({'question': state['question']})
    state['route'] = d.route
    state['trace'] = state.get('trace', []) + [f'classify_query→{d.route}']
    return state


def direct_answer(state: AdaptiveState) -> AdaptiveState:
    # 검색 없이 LLM 일반 지식으로만 응답 — 약관은 건드리지 않는다
    state['documents'] = []
    state['trace'].append('direct_answer')
    return state


def single_rag(state: AdaptiveState) -> AdaptiveState:
    state['documents'] = retriever.invoke(state['question'])
    state['trace'].append(f'single_rag(k={len(state["documents"])})')
    return state


# ────────────────────────────────────────────────
# Multi-hop 설계 원칙
# ────────────────────────────────────────────────
# 왜 "후속 검색어 **하나만**" 생성할까?
# → Adaptive RAG 의 multi_retrieval 경로는 "첫 검색이 놓친 **정보 공백 한 개**"만 채우는
#    가장 단순한 확장이다. 공백이 여러 개라면 이미 multi_retrieval 수준을 넘어서
#    **질문 분해 후 에이전트가 도구를 여러 번 호출**하는 별도 패턴(예: Agentic RAG)의
#    영역이 된다.
# → 즉 multi_hop_rag 는 "한 번 더 던져주면 정답이 나오는" 구간용 도구다.
# ────────────────────────────────────────────────
hop_prompt = ChatPromptTemplate.from_template(
    '원 질문: {q}\n'
    '1차 검색 결과:\n{ctx}\n\n'
    '위 결과에서 아직 답하지 못한 정보를 얻기 위한 **후속 검색어 하나**만 출력하세요. '
    '한 줄만 출력하고, 따옴표/설명 없이 검색어만 쓰세요.'
)
hop_chain = hop_prompt | llm | StrOutputParser()

def multi_hop_rag(state: AdaptiveState) -> AdaptiveState:
    # Hop 1
    docs1 = retriever.invoke(state['question'])
    ctx1 = '\n'.join(d.page_content for d in docs1)
    # Hop 2 — LLM이 생성한 서브쿼리로 추가 검색
    followup = hop_chain.invoke({'q': state['question'], 'ctx': ctx1}).strip()
    docs2 = retriever.invoke(followup)
    # 중복 제거
    seen, merged = set(), []
    for d in docs1 + docs2:
        key = d.page_content[:40]
        if key not in seen:
            seen.add(key)
            merged.append(d)
    state['documents'] = merged
    state['trace'].append(f'multi_hop_rag(hop1={len(docs1)}, followup={followup!r}, total={len(merged)})')
    return state


gen_prompt = ChatPromptTemplate.from_template(
    '아래 컨텍스트를 참고해 한국어로 간결히 답하세요. '
    '컨텍스트가 비어 있으면 일반 지식으로 답하되 "(근거 없음)" 문구를 끝에 붙이세요.\n\n'
    '[컨텍스트]\n{ctx}\n\n[질문] {q}\n[답변]'
)
gen_chain = gen_prompt | llm | StrOutputParser()

def generate(state: AdaptiveState) -> AdaptiveState:
    ctx = '\n---\n'.join(d.page_content for d in state['documents']) if state['documents'] else ''
    state['generation'] = gen_chain.invoke({'ctx': ctx, 'q': state['question']})
    state['trace'].append('generate')
    return state


def route_selector(state: AdaptiveState) -> str:
    return state['route']

In [ ]:
# [학습용] multi_hop_rag 가 실제로 어떤 순서로 정보를 모으는지 중간 상태를 모두 출력
demo_q = '삼성전자 자회사가 발행한 상품의 청약철회 기간은?'

docs1 = retriever.invoke(demo_q)
print(f'[Hop 1] 원 질문으로 1차 검색 → {len(docs1)}개 문서')
for d in docs1:
    print(f'   - [{d.metadata.get("article"):>8s}] {d.page_content[:50]}...')

ctx1 = '\n'.join(d.page_content for d in docs1)
followup = hop_chain.invoke({'q': demo_q, 'ctx': ctx1}).strip()
print(f'\n[LLM이 발견한 정보 공백 → 생성한 후속 검색어]')
print(f'   "{followup}"')

docs2 = retriever.invoke(followup)
print(f'\n[Hop 2] 후속 검색어로 2차 검색 → {len(docs2)}개 문서')
for d in docs2:
    print(f'   - [{d.metadata.get("article"):>8s}] {d.page_content[:50]}...')

print('\n→ 1차에서는 "자회사" 그룹구조만 걸리고 상품 약관은 부족할 때,')
print('  LLM이 "어떤 자회사?"라는 공백을 감지해 두 번째 검색으로 상품안내를 연결한다.')

In [ ]:
# ============================================================
# 품질 보강 루프: single_rag 뒤에만 얹는 "자가 채점 + 쿼리 재작성"
# ============================================================
# 흐름: single_rag → grade_documents
#        ├─ 관련 문서 있음 → generate
#        └─ 전부 무관     → transform_query → single_rag (1회에 한해)
# ============================================================

class DocumentRelevance(BaseModel):
    """검색 문서가 질문과 관련 있는지 판정한다."""
    relevant: Literal['yes', 'no'] = Field(description='문서가 질문에 답할 단서를 담고 있으면 yes.')
    reason: str = Field(description='판정 근거 1문장.')

grade_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 검색 품질 평가자입니다. 주어진 문서가 질문에 대답하는 데 쓸 만한 단서를 담고 있는지 평가하세요. '
     '정답을 완전히 포함할 필요는 없고, 관련 사실·용어·수치가 등장하면 yes 입니다.'),
    ('user', '[질문]\n{question}\n\n[문서]\n{doc}'),
])
grade_chain = grade_prompt | llm.with_structured_output(DocumentRelevance)

rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 검색 품질 개선 전문가입니다. 다음 질문이 벡터 검색으로 적절한 약관 조항을 찾지 못했습니다. '
     '핵심 키워드를 풀어쓰고, 약관·금융 도메인 용어를 보강해 한 문장으로 재작성하세요. '
     '질문 자체만 출력하고, 따옴표·설명·접두어 없이 한 줄로 쓰세요.'),
    ('user', '원 질문: {question}\n실패 이유: {reason}'),
])
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def grade_documents(state: AdaptiveState) -> AdaptiveState:
    """검색 문서를 한 개씩 평가해 relevant=yes 인 것만 남긴다."""
    kept, reasons = [], []
    for d in state['documents']:
        r = grade_chain.invoke({'question': state['question'], 'doc': d.page_content})
        if r.relevant == 'yes':
            kept.append(d)
        else:
            reasons.append(r.reason)
    state['documents'] = kept
    state['needs_rewrite'] = (len(kept) == 0) and (state.get('rewrite_count', 0) < 1)
    tag = 'keep' if kept else ('rewrite' if state['needs_rewrite'] else 'give_up')
    state['trace'].append(f'grade_documents({len(kept)} kept, action={tag})')
    # 재작성 분기에서 쓸 수 있도록 마지막 실패 근거를 state에 잠시 보관
    state['_last_grade_reason'] = reasons[0] if reasons else '관련 조항을 찾지 못함'
    return state


def transform_query(state: AdaptiveState) -> AdaptiveState:
    """원 질문을 재작성해 single_rag 재시도에 사용한다."""
    new_q = rewrite_chain.invoke({
        'question': state['question'],
        'reason': state.get('_last_grade_reason', ''),
    }).strip().strip('"').strip("'")
    state['trace'].append(f'transform_query({state["question"]!r} → {new_q!r})')
    state['question'] = new_q
    state['rewrite_count'] = state.get('rewrite_count', 0) + 1
    return state


def after_grade(state: AdaptiveState) -> str:
    """grade 결과에 따라 다음 노드를 결정."""
    return 'transform_query' if state.get('needs_rewrite') else 'generate'

print('grade_documents·transform_query 노드 정의 완료')

In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(AdaptiveState)
g.add_node('classify', classify_query)
g.add_node('direct', direct_answer)
g.add_node('single', single_rag)
g.add_node('multi', multi_hop_rag)
g.add_node('grade', grade_documents)
g.add_node('rewrite', transform_query)
g.add_node('generate', generate)

g.set_entry_point('classify')

# classify → 3갈래 분기
g.add_conditional_edges('classify', route_selector, {
    'no_retrieval': 'direct',
    'single_retrieval': 'single',
    'multi_retrieval': 'multi',
})

# direct / multi 는 바로 generate 로 수렴
g.add_edge('direct', 'generate')
g.add_edge('multi', 'generate')

# single 경로만 품질 보강 루프: single → grade → (rewrite → single) | generate
g.add_edge('single', 'grade')
g.add_conditional_edges('grade', after_grade, {
    'transform_query': 'rewrite',
    'generate': 'generate',
})
g.add_edge('rewrite', 'single')  # 재작성된 질문으로 single 재시도

g.add_edge('generate', END)

adaptive_app = g.compile()
print('Adaptive RAG 그래프 컴파일 완료 (classify + 3갈래 + grade/rewrite 루프)')

In [ ]:
# 내가 조립한 그래프가 실제로 이렇게 생겼는지 시각적으로 확인한다.
from IPython.display import Image, Markdown, display

mermaid_src = adaptive_app.get_graph().draw_mermaid()
print('--- Mermaid 소스 (마크다운 뷰어에 붙여넣으면 렌더됨) ---')
print(mermaid_src)

# 주피터에서 곧바로 렌더
display(Markdown(f'```mermaid\n{mermaid_src}\n```'))

# 외부 네트워크(Mermaid.ink) 또는 pyppeteer 로컬 렌더가 가능한 환경이면 PNG 도 표시
try:
    png_bytes = adaptive_app.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
except Exception as e:
    print(f'\n(PNG 이미지 렌더는 선택 사항 — 현재 환경에서 건너뜀: {type(e).__name__})')

## 5. 벤치마크 — 라우팅·품질 보강 루프가 실제로 동작하는지 검증

이 노트북이 테스트하려는 것은 두 가지다.

1. **라우터가 질문을 올바른 경로로 보내는가?** (3-way 분류 정확도)
2. **single 경로 뒤의 자가 채점·쿼리 재작성 루프가 실패 질의를 실제로 되살리는가?** (trace에 `transform_query` 가 찍히는지)

### 평가 질문 구성 (총 8개)

| 그룹 | 개수 | 기대 경로 | 노림수 |
|------|------|-----------|--------|
| 단일 사실 질의 (정상) | 5 | `single_retrieval` | 라우터가 single로 보내고, grade 가 통과시켜 바로 generate |
| 단일 사실 질의 (너무 짧음) | 1 | `single_retrieval` | 첫 검색이 빈약해 `transform_query` 로 1회 재작성 후 성공 |
| 일상 대화 | 1 | `no_retrieval` | 약관 건드리지 않고 direct_answer → generate |
| 엔티티 체인 | 1 | `multi_retrieval` | multi_hop_rag 에서 hop1 → 후속 검색 → hop2 |

> 🎯 **핵심 관찰 지점**: `trace` 컬럼에 `transform_query` 가 등장하는 행이 최소 1개 나오면 품질 보강 루프가 실효성 있게 작동한 것이다.

In [ ]:
# 약관 단일 조항으로 답할 수 있는 정상 질문 5개 (single_retrieval 기대)
capstone_qs = [
    '청약철회 기간은 며칠인가?',
    '카드 등 접근매체 분실 시 신고 절차는 어떻게 되나?',
    '수수료를 인상하려면 며칠 전에 공지해야 하는가?',
    '거래 내역에 이의가 있을 때 언제까지 이의신청을 할 수 있고, 금융회사의 회신 기한은 얼마인가?',
    '서비스 이용시간은 어떻게 되며, 일시 제한 사유에는 무엇이 있는가?',
]

# 품질 보강 루프 유발용: 지나치게 짧아 첫 검색이 부족할 가능성이 큰 질문
challenge_qs = [
    ('철회 며칠?', 'single_retrieval'),
]

# 다른 경로용
extra_qs = [
    ('오늘 날씨 어때', 'no_retrieval'),
    ('삼성전자의 자회사가 발행한 상품의 약관에 따른 청약철회 기간은?', 'multi_retrieval'),
]

all_qs = (
    [(q, 'single_retrieval') for q in capstone_qs]
    + challenge_qs
    + extra_qs
)

print('=== 평가할 질문 ({}개) ==='.format(len(all_qs)))
for i, (q, expected) in enumerate(all_qs, 1):
    short = q if len(q) < 55 else q[:52] + '…'
    print(f'  {i:>2}. [{expected:>17s}] {short}')

In [ ]:
import pandas as pd

rows = []
for q, expected in all_qs:
    init: AdaptiveState = {
        'question': q,
        'route': '',
        'documents': [],
        'generation': '',
        'trace': [],
        'needs_rewrite': False,
        'rewrite_count': 0,
    }
    final = adaptive_app.invoke(init)
    rows.append({
        '질문': q if len(q) < 50 else q[:47] + '…',
        '기대 경로': expected,
        '실제 경로': final['route'],
        '일치': '✅' if final['route'] == expected else '❌',
        '문서 수': len(final['documents']),
        '재작성': final.get('rewrite_count', 0),
        'trace 길이': len(final['trace']),
        '답변 길이': len(final['generation']),
    })

df = pd.DataFrame(rows)
df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (1) 실제 라우트 분포
route_counts = df['실제 경로'].value_counts().reindex(
    ['no_retrieval', 'single_retrieval', 'multi_retrieval'], fill_value=0
)
axes[0].bar(route_counts.index, route_counts.values, color=['#6baed6', '#74c476', '#fd8d3c'])
axes[0].set_title('실제 라우트 분포 (8개 질문)')
axes[0].set_ylabel('질문 수')
for i, v in enumerate(route_counts.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# (2) 재작성 루프가 몇 번 발동했나
rewrite_counts = df['재작성'].value_counts().sort_index()
axes[1].bar(rewrite_counts.index.astype(str), rewrite_counts.values, color='#9e9ac8')
axes[1].set_title('질문별 쿼리 재작성 횟수')
axes[1].set_xlabel('rewrite_count')
axes[1].set_ylabel('질문 수')
for i, v in enumerate(rewrite_counts.values):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# 경로별 일치율 요약
hit_rate = df.groupby('기대 경로').apply(lambda g: (g['일치'] == '✅').mean()).rename('일치율')
print('\n=== 기대 경로별 라우팅 일치율 ===')
print(hit_rate.to_string())

In [ ]:
# 특정 질문의 상세 trace / 검색 문서 / 답변을 한번에 확인
sample_q = '삼성전자의 자회사가 발행한 상품의 약관에 따른 청약철회 기간은?'
sample = adaptive_app.invoke({
    'question': sample_q,
    'route': '',
    'documents': [],
    'generation': '',
    'trace': [],
    'needs_rewrite': False,
    'rewrite_count': 0,
})

# 1) trace 를 step 번호가 붙은 표로
from IPython.display import display
trace_df = pd.DataFrame(
    [{'step': i + 1, 'node': step} for i, step in enumerate(sample['trace'])]
)
print('=== TRACE (상태 배낭이 거쳐간 노드들) ===')
display(trace_df)

# 2) 최종 검색 컨텍스트
print(f'\n=== 검색된 문서 ({len(sample["documents"])}개) ===')
for d in sample['documents']:
    print(f'  [{d.metadata.get("article")}] {d.page_content[:70]}...')

# 3) 최종 답변
print(f'\n=== 최종 답변 ===\n{sample["generation"]}')

## 6. Self-RAG(05) vs Adaptive RAG(06) 비교

| 관점 | Self-RAG (05) | Adaptive RAG (06) |
|------|--------------------|---------------------|
| **결정 시점** | 검색 **후** 평가 — 부족하면 재작성·재검색 | 검색 **전** 경로 결정 후 그 경로로 고정 |
| **라우팅 기준** | 문서별 `yes/no` 관련성 | 질문의 **복잡도**(no / single / multi) |
| **제어 흐름** | `retrieve → grade → rewrite → retrieve` 루프 | `classify → 3갈래 분기 → 경로별 처리 → generate` |
| **감사 용이성** | 루프 횟수가 가변 → trace 길이가 질문마다 다름 | 경로가 유한·예측 가능 → `trace` 가 결정론적 |
| **비용 예측** | retries 상한으로 제어 | 경로별 상한이 설계 시점에 고정 |
| **적합 상황** | 질문 유형은 비슷하지만 검색 품질이 들쭉날쭉할 때 | 질문 유형 자체가 다양하고 경로별 비용 차이가 클 때 |
| **실패 모드** | rewrite 가 엉뚱한 질문으로 바뀜 → 계속 재시도 | 라우터 오분류 → 엉뚱한 경로로 고정 |

> 🏦 **실무 팁**: 금융권 초기 배포에는 **Adaptive RAG**로 감사성과 비용 상한을 먼저 확보한 뒤, 각 경로(특히 `single_retrieval`) 안에서 **Self-RAG 스타일 grade·rewrite 루프**를 꽂는 조합이 가장 현실적이다. 이 노트북의 `single → grade → transform_query → single` 구조가 바로 그 조합이다.

## 7. Day 2 종합 의사결정 트리 — "언제 어느 기법을 쓰나"

Day 2 의 6개 노트북을 모두 마친 시점에, 새로운 RAG 요구사항을 받으면 아래 순서로 판단한다.

| # | 제목 | 핵심 | 쓰는 시점 |
|---|------|------|----------|
| 01 | Hybrid Ensemble | BM25 + dense + RRF | 정확한 용어 질의와 의미 질의가 섞일 때 (recall 확보) |
| 02 | Reranking | 2-stage (dense → cross-encoder) | **언제나 기본 탑재** — top-k 정확도가 성능의 80%를 좌우 |
| 03 | LangGraph 기초 | State · Node · Edge · 조건부 엣지 | 04·05·06의 공통 뼈대 |
| 04 | Naive RAG + 관련성 체크 | 조건부 엣지로 자가교정의 첫 단추 | Naive RAG가 왜 흔들리는지 · 조건부 엣지가 왜 필요한지 체감 |
| 05 | Self-RAG | `grade → rewrite → retrieve` 루프 | 검색 품질이 불안정하고 재시도가 필요할 때 |
| 06 | Adaptive RAG | 쿼리 복잡도 라우터 | 감사·비용 예측이 중요한 **운영 배포**의 진입점 |

```mermaid
flowchart TD
    START[새 RAG 요구사항] --> Q1{질문 유형이 다양한가?}
    Q1 -- 예 --> ADAPTIVE[06 Adaptive RAG]
    Q1 -- 아니오 --> Q2{검색 품질이 불안정한가?}
    Q2 -- 예 --> SELF[05 Self-RAG]
    Q2 -- 아니오 --> BASE[01 Hybrid + 02 Rerank]
    ADAPTIVE -.기본 탑재.-> BASE
    SELF -.기본 탑재.-> BASE
```

> 💡 **정리**: 01·02는 **검색 품질 자체**를 끌어올리는 기법이라 어떤 구조에서도 기본으로 깔고 간다. 그 위에 04·05·06은 **워크플로**를 얹어 가변적인 실패 상황에 대응한다. 실무 파이프라인은 대개 "01+02+06" 위에 경로별로 05 스타일의 루프를 섞는 모습이 된다.